# Dataset Training — Various methods tested



**This notebook tries the following:**
- EfficientNet-B0 (better for smaller inputs than ResNet-18)
- Mixup ON (alpha=0.3) — was accidentally left off last time
- Focal loss + sqrt weights + label smoothing (kept from last run)
- Heavier dropout + stronger weight decay to fight overfitting
- Stochastic Weight Averaging (SWA) in the final epochs for flatter minima
- Test-time augmentation (TTA)
- Comparison table at the end

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR
from torchvision import models
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, f1_score)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [ ]:
DATA_DIR   = '/content/drive/MyDrive/dataset_preprocessed/'
SAVE_DIR   = '/content/drive/MyDrive/output_models/'
os.makedirs(SAVE_DIR, exist_ok=True)

PHASE1_EPOCHS = 15
PHASE2_EPOCHS = 45
SWA_START     = 35     # start SWA averaging after this many total epochs
PHASE1_LR     = 2e-3
PHASE2_LR     = 3e-4
SWA_LR        = 1e-5
BATCH_SIZE    = 32
WEIGHT_DECAY  = 5e-4   # stronger than before
LABEL_SMOOTH  = 0.1
FOCAL_GAMMA   = 2.0
MIXUP_ALPHA   = 0.3    # MIXUP IS ON THIS TIME
TTA_PASSES    = 7

with open(os.path.join(DATA_DIR, 'metadata.json')) as f:
    meta = json.load(f)

NUM_CLASSES = meta['num_classes']
IDX_TO_LABEL = {int(k): v for k, v in meta['idx_to_label'].items()}
CLASS_NAMES = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]

# sqrt weights, normalized
raw_w = [meta['class_weights'][str(i)] for i in range(NUM_CLASSES)]
sqrt_w = [np.sqrt(w) for w in raw_w]
s = sum(sqrt_w)
norm_w = [w * NUM_CLASSES / s for w in sqrt_w]
class_weights = torch.FloatTensor(norm_w).to(device)

print(f'Classes: {NUM_CLASSES} — {CLASS_NAMES}')
print(f'Mixup: ON (alpha={MIXUP_ALPHA})')
print(f'SWA: starts at epoch {SWA_START}')
print(f'Weight decay: {WEIGHT_DECAY} (stronger to fight overfitting)')
print(f'Weights: {[f"{w:.2f}" for w in norm_w]}')

Classes: 5 — ['COPD', 'Healthy', 'Other Chronic', 'Pneumonia', 'Resp. Infection']
Mixup: ON (alpha=0.3)
SWA: starts at epoch 35
Weight decay: 0.0005 (stronger to fight overfitting)
Weights: ['0.36', '1.07', '1.17', '1.15', '1.24']


---
## 1 — Focal loss + Dataset

In [ ]:
class FocalLoss(nn.Module):
    """focuses on hard-to-classify samples instead of blanket reweighting"""
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.ls = label_smoothing

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight,
                             label_smoothing=self.ls, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def spec_augment(spec, nf=1, fw=15, nt=1, tw=25):
    aug = spec.copy()
    _, nm, nfr = aug.shape
    for _ in range(nf):
        f = np.random.randint(1, min(fw, nm)); f0 = np.random.randint(0, nm-f)
        aug[:, f0:f0+f, :] = 0.0
    for _ in range(nt):
        t = np.random.randint(1, min(tw, nfr)); t0 = np.random.randint(0, nfr-t)
        aug[:, :, t0:t0+t] = 0.0
    return aug


class ICBHIDataset(Dataset):
    def __init__(self, npz_path, augment=False):
        data = np.load(npz_path)
        self.specs = data['spectrograms']
        self.labels = data['disease_labels']
        self.augment = augment

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        spec = self.specs[idx].copy()
        if self.augment:
            spec = spec_augment(spec)
            if np.random.random() > 0.5:
                spec = spec[:, :, ::-1].copy()  # time flip
        return torch.FloatTensor(spec), torch.tensor(self.labels[idx], dtype=torch.long)


train_ds = ICBHIDataset(os.path.join(DATA_DIR, 'train_data.npz'), augment=True)
val_ds   = ICBHIDataset(os.path.join(DATA_DIR, 'val_data.npz'),   augment=False)
test_ds  = ICBHIDataset(os.path.join(DATA_DIR, 'test_data.npz'),  augment=False)
test_ds_tta = ICBHIDataset(os.path.join(DATA_DIR, 'test_data.npz'), augment=True)

# weighted sampler with sqrt weights
tr_labels = np.load(os.path.join(DATA_DIR, 'train_data.npz'))['disease_labels']
sw = np.array([np.sqrt(meta['class_weights'][str(int(l))]) for l in tr_labels])
sampler = WeightedRandomSampler(torch.DoubleTensor(sw), len(tr_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

Train: 5960 | Val: 964 | Test: 859


---
## 2 — EfficientNet-B0

In [ ]:
def build_efficientnet(num_classes):
    """
    EfficientNet-B0 — better suited for our spectrogram size (128x157)
    than ResNet-18. It uses compound scaling and is more parameter-efficient.
    """
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    # the classifier is model.classifier (a Sequential with dropout + linear)
    nf = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(nf, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    return model


def build_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    nf = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(nf, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    return model


# build both — we'll train both and ensemble them
model_eff = build_efficientnet(NUM_CLASSES).to(device)
model_res = build_resnet18(NUM_CLASSES).to(device)

print(f'EfficientNet-B0: {sum(p.numel() for p in model_eff.parameters()):,} params')
print(f'ResNet-18:       {sum(p.numel() for p in model_res.parameters()):,} params')
print(f'\nWe train both and ensemble their predictions at the end.')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 146MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 91.7MB/s]


EfficientNet-B0: 4,336,769 params
ResNet-18:       11,309,125 params

We train both and ensemble their predictions at the end.


---
## 3 — Training functions

In [ ]:
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam


def train_epoch(model, loader, criterion, optimizer, use_mixup=True, alpha=0.3):
    model.train()
    tl, cor, tot = 0., 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if use_mixup:
            xm, ya, yb, lam = mixup_data(x, y, alpha)
            pred = model(xm)
            loss = lam * criterion(pred, ya) + (1-lam) * criterion(pred, yb)
        else:
            pred = model(x)
            loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()*x.size(0)
        _, p = pred.max(1); tot += y.size(0); cor += p.eq(y).sum().item()
    return tl/tot, 100.*cor/tot


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tl, cor, tot = 0., 0, 0
    ap, al = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        tl += loss.item()*x.size(0)
        _, p = pred.max(1); tot += y.size(0); cor += p.eq(y).sum().item()
        ap.extend(p.cpu().numpy()); al.extend(y.cpu().numpy())
    acc = 100.*cor/tot
    f1 = 100.*f1_score(al, ap, average='macro', zero_division=0)
    return tl/tot, acc, f1, ap, al


@torch.no_grad()
def get_logits(model, loader):
    """get raw logits from a model for ensemble averaging"""
    model.eval()
    all_logits, all_labels = [], []
    for x, y in loader:
        x = x.to(device)
        all_logits.append(model(x).cpu())
        all_labels.extend(y.numpy())
    return torch.cat(all_logits, dim=0), np.array(all_labels)


@torch.no_grad()
def get_tta_logits(model, dataset, n_passes=7, batch_size=32):
    """run model n_passes times with augmentation, average logits"""
    model.eval()
    n = len(dataset)
    total = torch.zeros(n, NUM_CLASSES)
    for _ in range(n_passes):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
        start = 0
        for x, y in loader:
            logits = model(x.to(device)).cpu()
            total[start:start+logits.size(0)] += logits
            start += logits.size(0)
    return total / n_passes


print('All training functions ready. Mixup is ON.')

All training functions ready. Mixup is ON.


---
## 4 — Train a single model (reusable function)

In [ ]:
def train_full(model, model_name, save_prefix):
    """
    full 2-phase training pipeline for any model.
    returns the trained model and its history.
    """
    criterion = FocalLoss(weight=class_weights, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH)
    hist = {'epoch':[], 'phase':[], 'tl':[], 'ta':[], 'vl':[], 'va':[], 'vf':[], 'lr':[]}
    best_f1, best_ep = 0., 0
    save_path = os.path.join(SAVE_DIR, f'{save_prefix}_best.pth')

    # --- phase 1: head only ---
    for p in model.parameters(): p.requires_grad = False
    # find the classifier head (different name for different architectures)
    if hasattr(model, 'fc'):
        head = model.fc
    elif hasattr(model, 'classifier'):
        head = model.classifier
    else:
        head = list(model.children())[-1]
    for p in head.parameters(): p.requires_grad = True

    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                     lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PHASE1_EPOCHS, eta_min=1e-5)

    print(f'\n{"="*55}')
    print(f'  Training {model_name}')
    print(f'{"="*55}')
    print(f'Phase 1: head only')
    print(f'{"Ep":>3s} {"TrL":>7s} {"TrA":>6s} {"VaL":>7s} {"VaA":>6s} {"VaF1":>6s}')

    for ep in range(1, PHASE1_EPOCHS+1):
        tl,ta = train_epoch(model, train_loader, criterion, opt, use_mixup=True, alpha=MIXUP_ALPHA)
        vl,va,vf,_,_ = evaluate(model, val_loader, criterion)
        sched.step()
        lr = opt.param_groups[0]['lr']
        for k,v in [('epoch',ep),('phase','head'),('tl',tl),('ta',ta),('vl',vl),('va',va),('vf',vf),('lr',lr)]:
            hist[k].append(v)
        m = ''
        if vf > best_f1:
            best_f1 = vf; best_ep = ep
            torch.save(model.state_dict(), save_path); m=' *'
        print(f'{ep:3d} {tl:7.4f} {ta:5.1f}% {vl:7.4f} {va:5.1f}% {vf:5.1f}%{m}')

    # --- phase 2: full fine-tune with mixup ---
    for p in model.parameters(): p.requires_grad = True
    print(f'\nPhase 2: full fine-tune + mixup')

    # differential LR
    if hasattr(model, 'fc'):
        backbone_params = [p for n,p in model.named_parameters() if 'fc' not in n]
        head_params = list(model.fc.parameters())
    else:
        backbone_params = [p for n,p in model.named_parameters() if 'classifier' not in n]
        head_params = list(model.classifier.parameters())

    opt = optim.AdamW([
        {'params': backbone_params, 'lr': PHASE2_LR * 0.1},
        {'params': head_params,     'lr': PHASE2_LR}
    ], weight_decay=WEIGHT_DECAY)

    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=1, eta_min=1e-6)

    # SWA model (averages weights over final epochs for flatter minima)
    swa_model = AveragedModel(model)
    swa_active = False

    patience, patience_limit = 0, 18

    print(f'{"Ep":>3s} {"TrL":>7s} {"TrA":>6s} {"VaL":>7s} {"VaA":>6s} {"VaF1":>6s}')

    for ep in range(PHASE1_EPOCHS+1, PHASE1_EPOCHS+PHASE2_EPOCHS+1):
        tl,ta = train_epoch(model, train_loader, criterion, opt, use_mixup=True, alpha=MIXUP_ALPHA)
        vl,va,vf,_,_ = evaluate(model, val_loader, criterion)
        sched.step()
        lr = opt.param_groups[1]['lr']

        # activate SWA after SWA_START epochs
        if ep >= SWA_START and not swa_active:
            swa_active = True
            print(f'  → SWA activated at epoch {ep}')
        if swa_active:
            swa_model.update_parameters(model)

        for k,v in [('epoch',ep),('phase','full'),('tl',tl),('ta',ta),('vl',vl),('va',va),('vf',vf),('lr',lr)]:
            hist[k].append(v)
        m = ''
        if vf > best_f1:
            best_f1 = vf; best_ep = ep; patience = 0
            torch.save(model.state_dict(), save_path); m=' *'
        else:
            patience += 1
        print(f'{ep:3d} {tl:7.4f} {ta:5.1f}% {vl:7.4f} {va:5.1f}% {vf:5.1f}%{m}')
        if patience >= patience_limit:
            print(f'  Early stop.'); break

    # update SWA batch norm
    if swa_active:
        print('Updating SWA batch norm...')
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
        # save SWA model too
        swa_path = os.path.join(SAVE_DIR, f'{save_prefix}_swa.pth')
        torch.save(swa_model.module.state_dict(), swa_path)

    print(f'\n{model_name} done. Best val F1: {best_f1:.1f}% at epoch {best_ep}')
    return model, swa_model if swa_active else None, hist, best_f1, best_ep, save_path

print('Training pipeline ready.')

Training pipeline ready.


---
## 5 — Train EfficientNet-B0

In [ ]:
model_eff, swa_eff, hist_eff, f1_eff, ep_eff, path_eff = train_full(
    model_eff, 'EfficientNet-B0', 'effnet')


  Training EfficientNet-B0
Phase 1: head only
 Ep     TrL    TrA     VaL    VaA   VaF1
  1  0.7358  20.2%  0.4349  11.0%  10.2% *
  2  0.6933  23.2%  0.4933  11.4%  10.7% *
  3  0.6986  22.5%  0.4401  23.0%  16.3% *
  4  0.6881  24.8%  0.4375  22.6%  16.3% *
  5  0.6904  24.1%  0.4560  24.3%  14.8%
  6  0.6643  25.5%  0.4383  29.5%  17.0% *
  7  0.6666  26.7%  0.4413  29.3%  16.9%
  8  0.6642  26.4%  0.4283  28.9%  15.5%
  9  0.6479  27.4%  0.4345  32.1%  17.3% *
 10  0.6526  26.6%  0.4728  21.2%  13.6%
 11  0.6515  28.3%  0.4490  27.2%  15.8%
 12  0.6388  29.4%  0.4445  28.7%  16.1%
 13  0.6460  27.8%  0.4266  31.5%  17.6% *
 14  0.6356  28.1%  0.4366  30.4%  16.9%
 15  0.6142  29.0%  0.4415  28.5%  16.5%

Phase 2: full fine-tune + mixup
 Ep     TrL    TrA     VaL    VaA   VaF1
 16  0.6164  30.0%  0.4146  37.0%  20.8% *
 17  0.6002  32.6%  0.4103  44.2%  21.8% *
 18  0.5792  34.6%  0.4097  46.9%  24.0% *
 19  0.5314  36.4%  0.3849  55.9%  25.8% *
 20  0.5108  38.7%  0.3895  50.6%  24

---
## 6 — Train ResNet-18

In [ ]:
model_res, swa_res, hist_res, f1_res, ep_res, path_res = train_full(
    model_res, 'ResNet-18', 'resnet')


  Training ResNet-18
Phase 1: head only
 Ep     TrL    TrA     VaL    VaA   VaF1
  1  0.2460  61.4%  0.3863  81.5%  39.6% *
  2  0.2636  58.4%  0.3759  81.3%  38.7%
  3  0.2370  58.8%  0.3684  80.3%  40.5% *
  4  0.2389  58.1%  0.4458  80.9%  38.9%
  5  0.2460  65.6%  0.3889  80.7%  39.9%
  6  0.2416  58.4%  0.4197  80.5%  39.5%
  7  0.2376  60.6%  0.4076  80.9%  40.6% *
  8  0.2278  58.1%  0.4274  82.6%  40.3%
  9  0.2443  61.2%  0.3981  81.2%  40.0%
 10  0.2391  59.9%  0.3820  81.4%  40.6%
 11  0.2383  58.3%  0.3843  82.3%  40.0%
 12  0.2046  61.4%  0.4150  81.0%  39.7%
 13  0.2365  67.3%  0.3776  81.7%  39.5%
 14  0.2241  59.2%  0.3880  81.7%  40.1%
 15  0.2210  62.8%  0.3933  81.3%  40.3%

Phase 2: full fine-tune + mixup
 Ep     TrL    TrA     VaL    VaA   VaF1
 16  0.2237  60.7%  0.4171  81.4%  37.7%
 17  0.2301  61.0%  0.3853  81.7%  41.4% *
 18  0.2466  59.1%  0.3978  81.5%  37.3%
 19  0.2225  64.8%  0.3598  81.4%  39.0%
 20  0.2224  55.3%  0.3694  82.5%  40.4%
 21  0.2116  56.

---
## 8 — Ensemble inference

Average the predictions from both models (and optionally their SWA versions + TTA). This is like getting a second opinion from a different doctor.

In [ ]:
# load best checkpoints
model_eff.load_state_dict(torch.load(path_eff))
model_res.load_state_dict(torch.load(path_res))

criterion = FocalLoss(weight=class_weights, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH)

# --- individual model results ---
print('Individual model results on test set:')
print('-'*55)

_, acc_e, f1_e, preds_e, labels = evaluate(model_eff, test_loader, criterion)
print(f'EfficientNet-B0:  Acc={acc_e:.1f}%, F1={f1_e:.1f}%')

_, acc_r, f1_r, preds_r, _ = evaluate(model_res, test_loader, criterion)
print(f'ResNet-18:        Acc={acc_r:.1f}%, F1={f1_r:.1f}%')

# --- ensemble: average logits from both models ---
logits_e, _ = get_logits(model_eff, test_loader)
logits_r, _ = get_logits(model_res, test_loader)
ens_logits = (logits_e + logits_r) / 2
ens_preds = ens_logits.argmax(dim=1).numpy()
ens_acc = 100. * (ens_preds == np.array(labels)).mean()
ens_f1 = 100. * f1_score(labels, ens_preds, average='macro', zero_division=0)
print(f'Ensemble (avg):   Acc={ens_acc:.1f}%, F1={ens_f1:.1f}%')

# --- ensemble + TTA ---
print(f'\nRunning TTA ({TTA_PASSES} passes per model)...')
tta_e = get_tta_logits(model_eff, test_ds_tta, n_passes=TTA_PASSES)
tta_r = get_tta_logits(model_res, test_ds_tta, n_passes=TTA_PASSES)
ens_tta = (tta_e + tta_r) / 2
tta_preds = ens_tta.argmax(dim=1).numpy()
true_labels = test_ds.labels
tta_acc = 100. * (tta_preds == true_labels).mean()
tta_f1 = 100. * f1_score(true_labels, tta_preds, average='macro', zero_division=0)
print(f'Ensemble + TTA:   Acc={tta_acc:.1f}%, F1={tta_f1:.1f}%')

# --- SWA ensemble (if available) ---
swa_results = []
if swa_eff is not None and swa_res is not None:
    swa_e_logits, _ = get_logits(swa_eff, test_loader)
    swa_r_logits, _ = get_logits(swa_res, test_loader)
    swa_ens = (swa_e_logits + swa_r_logits) / 2
    swa_preds = swa_ens.argmax(dim=1).numpy()
    swa_acc = 100. * (swa_preds == np.array(labels)).mean()
    swa_f1 = 100. * f1_score(labels, swa_preds, average='macro', zero_division=0)
    print(f'SWA Ensemble:     Acc={swa_acc:.1f}%, F1={swa_f1:.1f}%')

    # mega ensemble: all 4 models
    mega = (logits_e + logits_r + swa_e_logits + swa_r_logits) / 4
    mega_preds = mega.argmax(dim=1).numpy()
    mega_acc = 100. * (mega_preds == np.array(labels)).mean()
    mega_f1 = 100. * f1_score(labels, mega_preds, average='macro', zero_division=0)
    print(f'Mega Ensemble:    Acc={mega_acc:.1f}%, F1={mega_f1:.1f}%')

Individual model results on test set:
-------------------------------------------------------
EfficientNet-B0:  Acc=64.6%, F1=45.3%
ResNet-18:        Acc=67.4%, F1=38.3%
Ensemble (avg):   Acc=67.8%, F1=43.5%

Running TTA (7 passes per model)...
Ensemble + TTA:   Acc=68.3%, F1=44.8%
SWA Ensemble:     Acc=68.1%, F1=42.4%
Mega Ensemble:    Acc=67.5%, F1=42.4%


In [ ]:
# save
results = {
    'effnet_acc': acc_e, 'effnet_f1': f1_e,
    'resnet_acc': acc_r, 'resnet_f1': f1_r,
    'ensemble_acc': ens_acc, 'ensemble_f1': ens_f1,
    'tta_acc': tta_acc, 'tta_f1': tta_f1,
    'best_method': best_name, 'best_acc': best_acc, 'best_f1': best_f1,
    'techniques': ['focal_loss','sqrt_weights','label_smoothing','mixup',
                   'warm_restarts','SWA','TTA','ensemble','time_flip','grad_clip'],
}
with open(os.path.join(SAVE_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('All saved to', SAVE_DIR)

All saved to /content/drive/MyDrive/ICBHI_models_maxperf/
